# Daily Spending per Platform

## Table

**`spending`** — records each user's spending on a given date and platform.

| Column | Type |
| --- | --- |
| `user_id` | int |
| `spend_date` | date |
| `platform` | varchar(10) |
| `amount` | int |

## Question

For each `spend_date`, report the **number of users** (`cnt`) and **total amount** (`amount`) per `platform`.

* If a user spent on **only one platform** on a date, count them under that platform.
* If a user spent on **both platforms** on a date, count them under `'both'` (not under individual platforms).
* Every date must include a `'both'` row — even if no users qualified, show `cnt = 0` and `amount = 0`.

Return `spend_date`, `platform`, `cnt`, and `amount`, ordered by `spend_date`.

### Approach
1. **Single-platform users** — group by date + user, keep rows where `count(distinct platform) = 1`, use `max(platform)`.
2. **Multi-platform users** — same grouping, keep rows where `count(distinct platform) > 1`, label platform as `'both'`.
3. **Dummy 'both' rows** — one per date with `NULL` user and `0` amount, so every date has a `'both'` entry.
4. **Union all three**, then group by date + platform to get `count(distinct user_id)` and `sum(amount)`.

In [0]:
%sql
-- ============================================================
-- Cell 1: Create and populate the `spending` table
-- ============================================================
create table b_sql.b_practice.spending 
(
user_id int,
spend_date date,
platform varchar(10),
amount int
);

insert into b_sql.b_practice.spending values(1,'2019-07-01','mobile',100),(1,'2019-07-01','desktop',100),(2,'2019-07-01','mobile',100)
,(2,'2019-07-02','mobile',100),(3,'2019-07-01','desktop',100),(3,'2019-07-02','desktop',100);


In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the spending table
-- ============================================================
select * from b_sql.b_practice.spending

In [0]:
%sql
-- ============================================================
-- Cell 3: SQL solution — daily spending per platform
-- Part 1: single-platform users (having count(distinct platform)=1)
-- Part 2: multi-platform users → 'both' (having count(distinct platform)>1)
-- Part 3: dummy 'both' rows per date (NULL user, 0 amount)
-- Final: group by date + platform → count(distinct user_id), sum(amount)
-- ============================================================
with spending as (select spend_date, user_id, max(platform) as platform , sum(amount) as amount from b_sql.b_practice.spending group by spend_date,user_id having count(distinct platform)=1
union all
select spend_date, user_id, 'both' as platform , sum(amount) as amount from b_sql.b_practice.spending group by spend_date,user_id having count(distinct platform)>1
union all 
select distinct spend_date, NULL as user_id, 'both' as platform , 0 as amount from b_sql.b_practice.spending
)
select spend_date,platform as platform,count(distinct user_id) as cnt,sum(amount) as amount from spending group by spend_date,platform order by spend_date

In [0]:
# ============================================================
# Cell 4: Load spending table into a PySpark DataFrame
# ============================================================
# ============================================================
# Cell 4: Load spending table into a PySpark DataFrame
# ============================================================
df_spend=spark.read.table("b_sql.b_practice.spending")
df_spend.display()

In [0]:
# ============================================================
# Cell 5: PySpark solution (part 1) — per-user aggregation
# df_one:        single-platform users (countDistinct == 1)
# df_both:       multi-platform users → 'both' (countDistinct > 1)
# df_both_dummy: one 'both' row per date (NULL user, 0 amount)
# ============================================================
# ============================================================
# Cell 5: PySpark solution (part 1) — per-user aggregation
# df_one:        single-platform users (countDistinct == 1)
# df_both:       multi-platform users → 'both' (countDistinct > 1)
# df_both_dummy: one 'both' row per date (NULL user, 0 amount)
# ============================================================
from pyspark.sql.functions import *

df_one=df_spend.groupBy("spend_date","user_id").agg(
    sum("amount").alias("amount"),max("platform").alias("platform"),
    countDistinct("platform").alias("platform_cnt")
    ).filter(col("platform_cnt")==1).drop("platform_cnt")
df_one.display()
df_both=df_spend.groupBy("spend_date","user_id").agg(
    sum("amount").alias("amount"),lit("both").alias("platform"),
    countDistinct("platform").alias("platform_cnt")
    ).filter(col("platform_cnt")==2).drop("platform_cnt")
df_both.display()
df_both_dummy=df_spend.select("spend_date",lit(None),
    lit(0).alias("amount"),lit("both").alias("platform")
    ).distinct()
df_both_dummy.display()

In [0]:
# ============================================================
# Cell 6: PySpark solution (part 2) — final aggregation
# Union all three parts, group by date + platform,
# count distinct users and sum amounts, order by date
# ============================================================
df_union=df_one.unionAll(df_both).unionAll(df_both_dummy)
df_final=df_union.groupBy("spend_date","platform").agg(
    sum("amount").alias("amount"),
    countDistinct("user_id").alias("user_cnt")
    ).orderBy("spend_date","platform")
df_final.display()
